## 0. Entorno y carga de ficheros

In [1]:
import pandas as pd
import openpyxl
import numpy as np
import seaborn as sns
import matplotlib as plt
import plotly.express as px
import plotly.io as pio
from pathlib import Path

In [ ]:
#Carga CSV 'FAOSTAT'
agric_ONU = pd.read_csv('/Users/rosamariasierraalmeria/Documents/GitHub/Impacto-del-clima-en-la-agricultura-/data/FAOSTAT_data_es_6-11-2026.csv')

In [ ]:
#Carga CSV 'NASA'
#skiprows= elimna las 13 primeras filas, que no son necesarias
#na_values=['-999', '-999.0'] convierte valor -999 y -999.0 en NA
clima_NASA = pd.read_csv(
    '/Users/rosamariasierraalmeria/Documents/GitHub/Impacto-del-clima-en-la-agricultura-/data/POWER_Point_Monthly_20050101_20241231_040d42N_003d70W_LST.csv',
    skiprows=13,
    na_values=['-999', '-999.0']
)

In [ ]:
#Carga de CSV 'ESYRCE'
#header=4 para llegar al encabezado que me interesa.
agric_ESP = pd.read_excel('/Users/rosamariasierraalmeria/Documents/GitHub/Impacto-del-clima-en-la-agricultura-/data/Dinaweb2024-2.xlsx', header=4)

Index(['Cultivos y cubiertas de suelo',                            2004,
                                  2005,                            2006,
                                  2007,                            2008,
                                  2009,                            2010,
                                  2011,                            2012,
                                  2013,                            2014,
                                  2015,                            2016,
                                  2017,                            2018,
                                  2019,                            2020,
                                  2021,                            2022,
                                  2023,                            2024],
      dtype='object')


In [28]:
#Carga de CSV 'tabla final AEMET'
clima_AEMET = pd.read_csv('/Users/rosamariasierraalmeria/Documents/GitHub/Impacto-del-clima-en-la-agricultura-/data/AEMET/tablas_limpias/aemet_final.csv')

---
## 1. Limpieza dataset.

In [9]:
#Gestión de nulos 'FAOSTAT'
agric_ONU.info()

<class 'pandas.DataFrame'>
RangeIndex: 5904 entries, 0 to 5903
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Código del ámbito          5904 non-null   str    
 1   Ámbito                     5904 non-null   str    
 2   Código del área (M49)      5904 non-null   int64  
 3   Área                       5904 non-null   str    
 4   Código del elemento        5904 non-null   int64  
 5   Elemento                   5904 non-null   str    
 6   Código del producto (CPC)  5904 non-null   str    
 7   Producto                   5904 non-null   str    
 8   Código del año             5904 non-null   int64  
 9   Año                        5904 non-null   int64  
 10  Unidad                     5904 non-null   str    
 11  Valor                      5823 non-null   float64
 12  Símbolo                    5904 non-null   str    
 13  Descripción del Símbolo    5904 non-null   str    
 14  Not

In [ ]:
#Total nulos 'FAOSTAT'
agric_ONU.isnull().sum()

Código del ámbito               0
Ámbito                          0
Código del área (M49)           0
Área                            0
Código del elemento             0
Elemento                        0
Código del producto (CPC)       0
Producto                        0
Código del año                  0
Año                             0
Unidad                          0
Valor                          81
Símbolo                         0
Descripción del Símbolo         0
Nota                         5694
dtype: int64

In [10]:
#Porcentaje de nulos 'FAOSTAT'
(agric_ONU.isnull().sum() / len(agric_ONU)) * 100

Código del ámbito             0.000000
Ámbito                        0.000000
Código del área (M49)         0.000000
Área                          0.000000
Código del elemento           0.000000
Elemento                      0.000000
Código del producto (CPC)     0.000000
Producto                      0.000000
Código del año                0.000000
Año                           0.000000
Unidad                        0.000000
Valor                         1.371951
Símbolo                       0.000000
Descripción del Símbolo       0.000000
Nota                         96.443089
dtype: float64

In [16]:
agric_ONU[agric_ONU["Valor"].isna()]["Producto"].value_counts()

Producto
Kenaf y otras fibras textiles de líber, brutas o enriadas    27
Grosellas                                                    14
Pelitre, flores secas                                        14
Café, verde                                                  13
Algarrobas                                                   12
Lino, en rama o enriado                                       1
Name: count, dtype: int64

Dataset con muy poco porcentaje de nulos, no merece la pena hacer modificaciones.

In [ ]:
#Revisión de valores duplicados 'FAOSTAT'.
agric_ONU.duplicated().sum()

np.int64(0)

In [ ]:
#Consultas de varias columnas para revisar el contenido y decidir si me las elimino o no.
agric_ONU["Símbolo"].value_counts(dropna=False)

Símbolo
A    4849
E     498
I     244
X     232
M      81
Name: count, dtype: int64

In [19]:
agric_ONU.groupby(
    ["Símbolo", "Descripción del Símbolo"]
).size()

Símbolo  Descripción del Símbolo                   
A        Cifra oficial                                 4849
E        Valor estimado                                 498
I        Valor imputado por una agencia receptora       244
M        Valor ausente; los datos no pueden existir      81
X        Cifra de fuente externa                        232
dtype: int64

In [36]:
#Gestión de nulos 'ESYRCE'
agric_ESP.info()

<class 'pandas.DataFrame'>
RangeIndex: 5494 entries, 0 to 5493
Data columns (total 22 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Cultivos y cubiertas de suelo  5494 non-null   str    
 1   2004                           3434 non-null   float64
 2   2005                           3417 non-null   float64
 3   2006                           3433 non-null   float64
 4   2007                           3484 non-null   float64
 5   2008                           3434 non-null   float64
 6   2009                           3527 non-null   float64
 7   2010                           3607 non-null   float64
 8   2011                           3609 non-null   float64
 9   2012                           3592 non-null   float64
 10  2013                           3661 non-null   float64
 11  2014                           3648 non-null   float64
 12  2015                           3711 non-null   float64
 13 

In [43]:
#Total nulos 'ESYRCE'
agric_ESP.isnull().sum()

Cultivos y cubiertas de suelo       0
2004                             2060
2005                             2077
2006                             2061
2007                             2010
2008                             2060
2009                             1967
2010                             1887
2011                             1885
2012                             1902
2013                             1833
2014                             1846
2015                             1783
2016                             1778
2017                             1721
2018                             1702
2019                             1686
2020                             1737
2021                             1758
2022                             1667
2023                             1651
2024                             1653
dtype: int64

In [ ]:
#Porcentaje de nulos 'ESYRCE'
(agric_ESP.isnull().sum() / len(agric_ESP)) * 100

Cultivos y cubiertas de suelo     0.000000
2004                             37.495450
2005                             37.804878
2006                             37.513651
2007                             36.585366
2008                             37.495450
2009                             35.802694
2010                             34.346560
2011                             34.310157
2012                             34.619585
2013                             33.363669
2014                             33.600291
2015                             32.453586
2016                             32.362577
2017                             31.325082
2018                             30.979250
2019                             30.688023
2020                             31.616309
2021                             31.998544
2022                             30.342191
2023                             30.050965
2024                             30.087368
dtype: float64

In [45]:
agric_ESP[agric_ESP[2024].isna()].head(20)

,Cultivos y cubiertas de suelo,2004,2005,2006,2007,2008,2009,2010,2011,2012,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
2,0115 CORUÑA (LA),416.9314,914.7639,196.2455,8.2086,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0132 ORENSE,NaN,172.9853,446.8761,57.2321,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.0071,NaN
5,0136 PONTEVEDRA,95.2970,95.2970,24.2977,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,"0233 P,DE ASTURIAS",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,32.3654,NaN,NaN,NaN,NaN
7,0401 ALAVA,NaN,16.5103,NaN,NaN,NaN,NaN,NaN,438.3591,12.2161,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,0626 LA RIOJA,22.8576,48.5013,NaN,NaN,NaN,NaN,NaN,NaN,93.9983,...,NaN,27.5505,NaN,6.6396,NaN,1359.1813,915.1304,NaN,139.6667,NaN
18,1005 AVILA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1521.4528,150.8037,359.5154,NaN,NaN,NaN,NaN,58.4679,NaN,NaN
21,1034 PALENCIA,NaN,NaN,NaN,NaN,NaN,NaN,198.0568,NaN,NaN,...,NaN,234.7944,536.8918,NaN,67.1869,135.5797,NaN,213.8905,282.8777,NaN
23,1040 SEGOVIA,NaN,NaN,149.9561,10.2994,NaN,48.1562,NaN,NaN,NaN,...,208.4872,233.6223,69.1278,NaN,34.2340,56.8111,60.0239,69.5693,12.9272,NaN
25,1047 VALLADOLID,317.4045,249.6652,443.6814,1369.0892,1871.2164,175.3399,1153.6909,52.5940,209.0087,...,48.0111,801.7481,3168.1527,1178.9208,1534.4748,430.8334,1060.2964,75.3029,273.1305,NaN


In [52]:
#Revisión de valores duplicados 'ESYRCE'.
agric_ESP.duplicated().sum()

np.int64(3)

In [53]:
agric_ESP.isnull().all(axis=1).sum()

np.int64(0)

In [54]:
agric_ESP["Cultivos y cubiertas de suelo"].isnull().sum()

np.int64(0)

In [55]:
agric_ESP["Cultivos y cubiertas de suelo"].duplicated().sum()

np.int64(5262)

In [56]:
agric_ESP.describe()

,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
count,3.434000e+03,3.417000e+03,3.433000e+03,3.484000e+03,3.434000e+03,3.527000e+03,3.607000e+03,3.609000e+03,3.592000e+03,3.661000e+03,...,3.711000e+03,3.716000e+03,3.773000e+03,3.792000e+03,3.808000e+03,3.757000e+03,3.736000e+03,3.827000e+03,3.843000e+03,3.841000e+03
mean,5.880933e+04,5.910192e+04,5.882646e+04,5.802016e+04,5.886654e+04,5.731433e+04,5.604270e+04,5.601164e+04,5.627673e+04,5.521607e+04,...,5.453360e+04,5.446022e+04,5.363769e+04,5.337004e+04,5.314661e+04,5.386960e+04,5.417298e+04,5.288499e+04,5.266469e+04,5.269266e+04
std,9.694622e+05,9.744602e+05,9.715076e+05,9.644911e+05,9.708913e+05,9.574625e+05,9.468885e+05,9.471060e+05,9.498576e+05,9.412848e+05,...,9.359118e+05,9.356724e+05,9.286521e+05,9.267123e+05,9.249863e+05,9.312058e+05,9.340932e+05,9.228362e+05,9.206436e+05,9.207206e+05
min,0.000000e+00,0.000000e+00,1.220000e-01,3.490000e-02,0.000000e+00,0.000000e+00,3.880000e-02,8.800000e-03,3.700000e-02,3.700000e-02,...,3.700000e-02,3.700000e-02,5.500000e-02,7.760000e-02,6.150000e-02,6.150000e-02,6.150000e-02,6.150000e-02,6.150000e-02,9.980000e-02
25%,7.056820e+01,6.583040e+01,6.011160e+01,5.898690e+01,6.210922e+01,5.981530e+01,5.433955e+01,5.436870e+01,6.096515e+01,5.493870e+01,...,5.291150e+01,5.391953e+01,5.221430e+01,5.495623e+01,5.151100e+01,5.626970e+01,5.643470e+01,4.852340e+01,5.466370e+01,6.457860e+01
50%,7.726043e+02,8.092666e+02,7.284277e+02,6.901402e+02,7.254279e+02,6.773252e+02,6.230448e+02,6.502478e+02,6.691887e+02,6.445216e+02,...,6.653940e+02,6.969135e+02,6.590150e+02,6.114331e+02,5.960557e+02,6.411999e+02,6.931721e+02,6.121755e+02,5.942679e+02,6.095705e+02
75%,9.024361e+03,9.366150e+03,9.229772e+03,8.912999e+03,8.706034e+03,8.562425e+03,8.017842e+03,8.380562e+03,8.485785e+03,8.058211e+03,...,7.957133e+03,8.091787e+03,7.605235e+03,7.487405e+03,7.592756e+03,7.980947e+03,8.401548e+03,7.674742e+03,7.363464e+03,7.412156e+03
max,5.048781e+07,5.048781e+07,5.048781e+07,5.053556e+07,5.053693e+07,5.053691e+07,5.053651e+07,5.053651e+07,5.053651e+07,5.053651e+07,...,5.059355e+07,5.059355e+07,5.059375e+07,5.059480e+07,5.059558e+07,5.059703e+07,5.059757e+07,5.059772e+07,5.059760e+07,5.059812e+07
